# Skill 40: LoRA (Low-Rank Adaptation)

**Phase 3 - LLMs & Generative AI Applications (Skills 31-45)**

## What this skill is about
Adapt a frozen linear layer by training a small low-rank update, and quantify the parameter saving.

## Objectives
- Explain low-rank updates and parameter efficiency.
- Freeze pretrained/base linear weights.
- Implement the base output plus scale x B(A(x)) and train only A and B.

Run the cells top to bottom. Requires `torch`, `numpy` and `matplotlib` (`pip install torch numpy matplotlib`).

## Theory and core formulas
$$y=xW^{\top}+\tfrac{\alpha}{r}(xA^{\top})B^{\top},\qquad \Delta W=\tfrac{\alpha}{r}BA$$
Trainable parameters: $r(d_{in}+d_{out})$ = 192 for 32->16, r=4. B starts at zero so the model begins identical to the base.

## Implementation
Reference code from the lab manual (unchanged).

In [ ]:
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank=4, alpha=8):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.A = nn.Parameter(torch.randn(rank, base.in_features) * 0.02)
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))
        self.scale = alpha / rank

    def forward(self, x):
        update = (x @ self.A.T) @ self.B.T
        return self.base(x) + self.scale * update

base = nn.Linear(32, 16)
layer = LoRALinear(base, rank=4, alpha=8)
x = torch.randn(10, 32)
target = torch.randn(10, 16)
opt = torch.optim.Adam([layer.A, layer.B], lr=1e-2)
for _ in range(10):
    opt.zero_grad()
    loss = ((layer(x) - target) ** 2).mean()
    loss.backward()
    opt.step()
print("loss:", float(loss))
print("trainable parameters:", sum(p.numel() for p in layer.parameters() if p.requires_grad))
print("base frozen:", all(not p.requires_grad for p in layer.base.parameters()))


## What to check
**Expected result:** Only 192 trainable parameters; the base layer stays frozen.

## Pitfalls
Freeze the base explicitly. Do not initialise B randomly. Merge with $W+\tfrac{\alpha}{r}BA$ for zero inference overhead.